In [2]:
import numpy as np
import matplotlib.pyplot as plt

# Принудительно устанавливаем стандартную светлую тему
plt.style.use('default')
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'text.usetex': False
})

def autocorr(x, max_lag=20):
    x_norm = x - np.mean(x)
    var = np.var(x)
    if var == 0:
        return np.zeros(max_lag + 1)
    corr = np.correlate(x_norm, x_norm, mode='full')
    corr = corr[len(x_norm) - 1:] / (var * len(x_norm))
    return corr[:max_lag + 1]

N = 100_000

# =======================================================
# 1. ЛОГИСТИЧЕСКОЕ ОТОБРАЖЕНИЕ
# =======================================================
z_log = np.zeros(N)
z_log[0] = 0.314159265
for k in range(N - 1):
    z_log[k + 1] = 4.0 * z_log[k] * (1.0 - z_log[k])

# 1.1 Гистограмма
fig, ax = plt.subplots(figsize=(5, 4), dpi=300, facecolor='white')
ax.hist(z_log, bins=100, density=True, color='skyblue', edgecolor='black', linewidth=0.3)
z_grid = np.linspace(0.001, 0.999, 500)
ax.plot(z_grid, 1.0 / (np.pi * np.sqrt(z_grid * (1.0 - z_grid))), 'r-', lw=2, label=r'$\rho^*(z)$')
ax.set_title(r'Плотность вероятности $\rho^*(z)$')
ax.set_xlabel('$z$')
ax.set_ylabel('Плотность')
ax.set_ylim(0, 5)
ax.legend()
plt.tight_layout()
plt.savefig('logistic_hist.pdf', facecolor='white', transparent=False)
plt.close()

# 1.2 Автокорреляция
fig, ax = plt.subplots(figsize=(5, 4), dpi=300, facecolor='white')
lags = np.arange(21)
ax.stem(lags, autocorr(z_log, 20), basefmt=" ")
ax.set_title(r'Автокорреляционная функция $R(m)$')
ax.set_xlabel('Лаг $m$')
ax.set_ylabel('$R(m)$')
ax.set_ylim(-0.1, 1.05)
plt.tight_layout()
plt.savefig('logistic_acf.pdf', facecolor='white', transparent=False)
plt.close()

# 1.3 Диаграмма возвратов
fig, ax = plt.subplots(figsize=(5, 4), dpi=300, facecolor='white')
ax.scatter(z_log[:2000], z_log[1:2001], s=2, color='darkblue', alpha=0.5)
ax.set_title(r'Диаграмма возвратов ($z_{k+1}$ от $z_k$)')
ax.set_xlabel('$z_k$')
ax.set_ylabel('$z_{k+1}$')
plt.tight_layout()
plt.savefig('logistic_return.pdf', facecolor='white', transparent=False)
plt.close()

# =======================================================
# 2. ОТОБРАЖЕНИЕ ПАЛАТКИ (С ЗАЩИТОЙ ОТ СХЛОПЫВАНИЯ В 0)
# =======================================================
z_tent = np.zeros(N)
z_tent[0] = 0.314159265
eps = 1e-15
for k in range(N - 1):
    zk = z_tent[k]
    # Добавляем микросдвиг от битового зануления при делении пополам
    if zk < 0.5:
        z_next = 2.0 * zk
    else:
        z_next = 2.0 * (1.0 - zk)
    if z_next <= 0.0 or z_next >= 1.0 or abs(z_next - 0.5) < eps:
        z_next = (z_next + 1e-7 * np.pi) % 1.0
    z_tent[k + 1] = z_next

# 2.1 Гистограмма
fig, ax = plt.subplots(figsize=(5, 4), dpi=300, facecolor='white')
ax.hist(z_tent, bins=100, density=True, color='lightgreen', edgecolor='black', linewidth=0.3)
ax.axhline(1.0, color='r', lw=2, label=r'$\rho^*(z) \equiv 1$')
ax.set_title(r'Плотность вероятности $\rho^*(z)$')
ax.set_xlabel('$z$')
ax.set_ylabel('Плотность')
ax.set_ylim(0, 2)
ax.legend()
plt.tight_layout()
plt.savefig('tent_hist.pdf', facecolor='white', transparent=False)
plt.close()

# 2.2 Автокорреляция
fig, ax = plt.subplots(figsize=(5, 4), dpi=300, facecolor='white')
ax.stem(lags, autocorr(z_tent, 20), basefmt=" ")
ax.set_title(r'Автокорреляционная функция $R(m)$')
ax.set_xlabel('Лаг $m$')
ax.set_ylabel('$R(m)$')
ax.set_ylim(-0.1, 1.05)
plt.tight_layout()
plt.savefig('tent_acf.pdf', facecolor='white', transparent=False)
plt.close()

# 2.3 Диаграмма возвратов
fig, ax = plt.subplots(figsize=(5, 4), dpi=300, facecolor='white')
ax.scatter(z_tent[:2000], z_tent[1:2001], s=2, color='darkgreen', alpha=0.5)
ax.set_title(r'Диаграмма возвратов ($z_{k+1}$ от $z_k$)')
ax.set_xlabel('$z_k$')
ax.set_ylabel('$z_{k+1}$')
plt.tight_layout()
plt.savefig('tent_return.pdf', facecolor='white', transparent=False)
plt.close()

# =======================================================
# 3. ОТОБРАЖЕНИЕ ЭНОНА
# =======================================================
x_hen = np.zeros(N)
y_hen = np.zeros(N)
x_hen[0], y_hen[0] = 0.1, 0.1
a, b = 1.4, 0.3
for k in range(N - 1):
    x_hen[k + 1] = 1.0 - a * (x_hen[k]**2) + y_hen[k]
    y_hen[k + 1] = b * x_hen[k]

# 3.1 Аттрактор (x, y)
fig, ax = plt.subplots(figsize=(5, 4), dpi=300, facecolor='white')
ax.scatter(x_hen[1000:15000], y_hen[1000:15000], s=0.4, color='purple', alpha=0.6)
ax.set_title(r'Странный аттрактор Энона')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
plt.tight_layout()
plt.savefig('henon_attractor.pdf', facecolor='white', transparent=False)
plt.close()

# 3.2 Гистограмма проекции x
fig, ax = plt.subplots(figsize=(5, 4), dpi=300, facecolor='white')
ax.hist(x_hen[1000:], bins=100, density=True, color='plum', edgecolor='black', linewidth=0.3)
ax.set_title(r'Плотность проекции $x_k$ (мера SRB)')
ax.set_xlabel('$x$')
ax.set_ylabel('Плотность')
plt.tight_layout()
plt.savefig('henon_hist.pdf', facecolor='white', transparent=False)
plt.close()

# 3.3 Диаграмма возвратов x_{k+1} от x_k
fig, ax = plt.subplots(figsize=(5, 4), dpi=300, facecolor='white')
ax.scatter(x_hen[1000:5000], x_hen[1001:5001], s=1, color='indigo', alpha=0.5)
ax.set_title(r'Диаграмма возвратов ($x_{k+1}$ от $x_k$)')
ax.set_xlabel('$x_k$')
ax.set_ylabel('$x_{k+1}$')
plt.tight_layout()
plt.savefig('henon_return.pdf', facecolor='white', transparent=False)
plt.close()

print("Все 9 графиков успешно сохранены в отдельные PDF на белом фоне!")

Все 9 графиков успешно сохранены в отдельные PDF на белом фоне!
